# Corpus project as notebook

# Create the document class

In [ ]:
import pdfminer.high_level as pdfhl
from pathlib import Path
import nltk
import os

class Document():
    def __init__(self, file_path):
        self.file_path = Path(file_path)
        self.filename = os.path.basename(file_path)
        self.text = self._extract_pdf_text()
        self.tokens = self._tokenize()

    def _check_file_type(self):
        if self.file_path.suffix.lower() == ".pdf":
            return True
        
    def _extract_pdf_text(self):
        #check wether pdf
        if not self._check_file_type():
            return

        # use pdf-miner to extract text
        text = pdfhl.extract_text(self.file_path)

        return text
    
    def _tokenize(self):
        lowered_text = self.text.lower()
        tokens = nltk.word_tokenize(lowered_text, preserve_line=True)

        return tokens

## Create the Corpus-class

In [ ]:
import os
from my_tools.Document import Document

class Corpus():
    def __init__(self, folder_path):
        self.folderpath = folder_path
        self.documents = self._load_documents()

    def _load_documents(self):
        documents = []
        
        for filename in os.listdir(self.folderpath):
            filepath = os.path.join(self.folderpath, filename)
            doc = Document(filepath)
            # add reaction if doc was not pdf
            documents.append(doc)

        return documents

    def get_document(self, filename): 
        for doc in self.documents: 
            if doc.filename == filename: 
                return doc 
            return None
    
    def get_all_tokens(self):
        tokens = []

        for doc in self.documents:
            tokens.extend(doc.tokens)

        return tokens

## Create TextProcessor-class

In [ ]:
import os
import string
from nltk.probability import FreqDist
from nltk.corpus import stopwords

with open("my_tools/stopwords.txt") as f:
    STOPWORDS = [line.strip() for line in f.readlines()]
PUNCT = set(string.punctuation)

class TextProcessor():

    @staticmethod
    def clean_tokens(tokens, remove_stopwords=False, remove_punct=False):
        cleaned = []

        for token in tokens:
            if remove_punct and all(ch in string.punctuation for ch in token): 
                continue
            if remove_stopwords and token.lower() in STOPWORDS: 
                continue
            cleaned.append(token)
        return cleaned

    @staticmethod
    def concordance(tokens, target, width=5):
        target = target.lower()
        results = []
        for i, token in enumerate(tokens): 
            if token == target: 
                left = " ".join(tokens[max(0, i - width): i]) 
                right = " ".join(tokens[i + 1: i + 1 + width]) 
                results.append((left, token, right))
        
        return results

    @staticmethod
    def ngrams(tokens, target, n):
        pass

    @staticmethod
    def freq_dist(tokens):
        return FreqDist(tokens)
    
    @staticmethod
    def token_freq(tokens, target):
        return tokens.count(target)

## Create StreamlitApp

In [ ]:
import os
import streamlit as st
import pandas as pd

class StreamlitApp():
    def __init__(self, corpus_path = "Corpus/raw_data/"):
        self.corpus = Corpus(corpus_path) 
        
        self.processor = TextProcessor()

        st.title = "Relinquishment Report Explorer"
        st.sidebar.write(f"Loaded {len(self.corpus.documents)} documents")

        self.mode = st.sidebar.radio( "Choose analysis mode", ["Concordance", "Word frequency", "Frequent words", "N-gram search", "Metadata viewer"] ) 
        
        self.run()

    def _doc_choise(self):
        doc_choice = st.selectbox( "Document", ["All documents"] + [doc.filename for doc in self.corpus.documents])
        return doc_choice

    def _get_doc_choice(self, doc_choice):
        if doc_choice == "All documents": 
            tokens = self.corpus.get_all_tokens() 
        else: 
            tokens = self.corpus.get_document(doc_choice).tokens
        
        return tokens


    def run(self):
        if self.mode == "Concordance":
            self.show_concordance()
        elif self.mode == "Frequent words":
            self.show_frequencies()
        elif self.mode == "Word frequency":
            self.show_word_frequency()

    def show_concordance(self):
        st.header("Concordance") 
        search_word = st.text_input("Search word:") 
        window = st.slider("Context window", 2, 20, 5)

        if not search_word: 
            st.info("Enter a word to search.")
            return

        doc_choice = self._doc_choise()
        tokens = self._get_doc_choice(doc_choice)

        results = TextProcessor.concordance(tokens, search_word, window)
        df = pd.DataFrame(results, columns=["left", "token", "right"])

        st.dataframe(df)

    def show_word_frequency(self):
        st.header("Word frequency")

        search_word = st.text_input("Search word:").lower()
        if not search_word: 
            st.info("Enter a word to search.") 
            return

        doc_choice = self._doc_choise()
        tokens = self._get_doc_choice(doc_choice)

        count = TextProcessor.token_freq(tokens=tokens, target=search_word)
        st.write(f"Frekvens for '{search_word}': {count}")
        


    def show_frequencies(self):
        st.header("Frequent words")

        doc_choice = st.selectbox( 
            "Document", ["All documents"] + [doc.filename for doc in self.corpus.documents] 
        )

        if doc_choice == "All documents": 
            tokens = self.corpus.get_all_tokens()
        else: 
            tokens = self.corpus.get_document(doc_choice).tokens

        remove_stop = st.checkbox("Exclude grammatical words (stopwords)") 
        remove_punct = st.checkbox("Exclude punctuation tokens")

        tokens = TextProcessor.clean_tokens(tokens, remove_stop, remove_punct)

        fdist = self.processor.freq_dist(tokens) 
        top_n = st.slider("Top N words", 10, 200, 50)

        items = fdist.most_common(top_n) 
        st.table({"Word": [w for w, _ in items], "Frequency": [f for _, f in items]})

    def show_n_grams(self):
        st.header("N-gram search") 
        search_word = st.text_input("Search word:") 
        n = st.slider("n (size of n-grams)", 2, 6, 3) 
        doc_choice = st.selectbox( "Document", ["All documents"] + [doc.filename for doc in self.corpus.documents] ) 
        if not search_word: 
            st.info("Enter a word to search.") 
            return 
        if doc_choice == "All documents": 
            tokens = self.corpus.all_tokens() 
        else: 
            tokens = self.corpus.get_document(doc_choice).tokens 
            hits = self.analyzer.ngram_search(tokens, search_word, n) 
            st.write(f"Found {len(hits)} matches") 
            for gram in hits[:100]: 
                st.markdown(" ".join(gram))